# Rendering text

This notebook uses Vazirmatn, the Persian, Arabic and Latin font of `data/fonts` (SIL OFL 1.1), so nothing else is needed. Only load fonts you trust: the font reader checks its bounds but has not been fuzzed.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import libstb as stb

DATA = Path.cwd().parents[1] / "data"      # data/ at the root of the repository (see data/README.md)
font = stb.Font.open(DATA / "fonts" / "Vazirmatn-Regular.ttf")
size = 40

## Metrics

In [ ]:
print(font.metrics(size))
print("advance A:", font.advance("A", size))
print("kerning A,V:", font.kerning("A", "V", size))
print("measure:", font.measure("Hello", size))

## A text bitmap

`render` takes UTF-8 (newlines are fine) and returns a 1-channel coverage image.

In [ ]:
text = font.render("Hello\nw\u00f6rld", size)
print(text.bitmap.width, text.bitmap.height, (text.origin_x, text.origin_y))
plt.imshow(text.bitmap.numpy()[:, :, 0], cmap="gray_r"); plt.axis("off");

## Single glyph

In [ ]:
g = font.render_glyph("A", 120)
print("offset", (g.x_offset, g.y_offset), "advance", g.advance)
plt.imshow(g.bitmap.numpy()[:, :, 0], cmap="gray_r"); plt.axis("off");

## Glyph atlas

A packed sheet of glyphs plus a lookup table, handy for game or GPU text.

In [ ]:
atlas = font.make_atlas("ABCDEFGabcdefg0123456789", 32, 256, 128)
plt.figure(figsize=(8, 4))
plt.imshow(atlas.image.numpy()[:, :, 0], cmap="gray_r"); plt.axis("off")
ag = atlas["A"]
print("A lives at", (ag.x0, ag.y0, ag.x1, ag.y1))

## Text on a picture, Persian included

`draw_text` returns a new image. Write Persian and Arabic as you would type it: joining forms, ligatures and the right-to-left order are done for you, and mixed text lays out in the right order.

In [ ]:
photo = stb.Image.open(DATA / "images" / "astronaut.png")
labeled = photo.draw_text("Hello, libstb", font, 40, 20, 16, color="white")
labeled = labeled.draw_text("\u0633\u0644\u0627\u0645 \u062f\u0646\u06cc\u0627", font, 56, 20, 90, color="#ffd166")
labeled = labeled.draw_text("libstb \u06cc\u0639\u0646\u06cc \u06a9\u062a\u0627\u0628\u062e\u0627\u0646\u0647", font, 32, 20, 170, color="#9be7ff")
plt.figure(figsize=(6, 6)); plt.imshow(labeled.numpy()); plt.axis("off");

## Glyphs, outlines and shaping

The glyph-level API gives the pieces in font units (y up): `shape` decides which glyphs, in which order and how far apart, and `outline` is each glyph's contour. Drawn through a `Matrix`, the text can be any size, angle or shear. `draw_text` scales by `size / (ascent - descent)`, so do the same to match it.

In [ ]:
d = font.design_metrics
s = 64 / (d.ascent - d.descent)                 # 64 px: the same scale draw_text uses for its size
shaped = font.shape("\u0633\u0644\u0627\u0645 AV")
print([(g.glyph, g.cluster) for g in shaped])   # visual order: the clusters run backwards in the Persian word

img = stb.Image(np.full((140, 420, 3), 250, np.uint8))
pen = 0.0
for g in shaped:
    # Each call applies to a point first, so read it from the bottom: scale and flip y, move along the line,
    # turn the whole line, then put it on the image.
    m = stb.Matrix().translate(20, 100).rotate(-0.2).translate((pen + g.x_offset) * s, 0).scale(s, -s)
    img = img.fill_outline(stb.Outline(font.outline(g.glyph)), "#1d3557", matrix=m)
    pen += g.x_advance                          # in font units
plt.imshow(img.numpy()); plt.axis("off");